In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import resample
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
import warnings
from math import sqrt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm

In [2]:
# load data
df_att = pd.read_csv('attendance data with features.csv')

rand_seed = 34

In [3]:
# remove unnecessary features
# duplicates, linearly dependent variables, etc.

remove = ['HGP', 'VGP',
          'HPTS', 'VPTS',
          'HGF', 'VGF',
          'HGA', 'VGA',
          'HPP%', 'VPP%', 
          'HPK%', 'HPK%',
          'HS', 'VS', 
          'HSA', 'VSA']

df_att = df_att.drop(columns = remove)

In [4]:
# A, PA, CAP
df_cap = df_att.copy().drop(columns = ['LA', 'LCAP'])

# LA, LCAP
df_lcap = df_att.copy().drop(columns = ['A', 'PA','CAP'])

X_cap_a = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'LCAP'])
y_cap_a = df_att['A']

X_cap_pa = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'LCAP'])
y_cap_pa = df_att['PA']

X_lcap = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'CAP'])
y_lcap = df_att['LA']

In [5]:
# Function to compute evaluation metrics with dependent variable transformation
def compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred):
    if dep == 'A':
        y_train_true, y_train_pred = y_train, y_train_pred
        y_test_true, y_test_pred = y_test, y_test_pred

    elif dep == 'LA':
        y_train_true, y_train_pred = np.exp(y_train), np.exp(y_train_pred)
        y_test_true, y_test_pred = np.exp(y_test), np.exp(y_test_pred)

    elif dep == 'PA':
        y_train_true, y_train_pred = y_train * X_train['CAP'], y_train_pred * X_train['CAP']
        y_test_true, y_test_pred = y_test * X_test['CAP'], y_test_pred * X_test['CAP']

    return {
        "RMSE Train": sqrt(mean_squared_error(y_train_true, y_train_pred)),
        "MAE Train": mean_absolute_error(y_train_true, y_train_pred),
        "R² Train": r2_score(y_train_true, y_train_pred),
        "RMSE Test": sqrt(mean_squared_error(y_test_true, y_test_pred)),
        "MAE Test": mean_absolute_error(y_test_true, y_test_pred),
        "R² Test": r2_score(y_test_true, y_test_pred)
    }

In [6]:
# Compute Pearson and Spearman correlation matrices
corr_matrix_pearson = df_att.drop(columns=['H', 'V']).corr(method='pearson')
corr_matrix_spearman = df_att.drop(columns=['H', 'V']).corr(method='spearman')

# Function to plot and save heatmap
def plot_heatmap(corr_matrix, title, save_path):
    plt.figure(figsize=(15, 10))
    sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, square=True)
    plt.title(title)
    plt.xticks(ticks=range(len(corr_matrix.columns)), labels=corr_matrix.columns, fontsize=7, rotation=90)
    plt.yticks(ticks=range(len(corr_matrix.index)), labels=corr_matrix.index, fontsize=7, rotation=0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')  # Save the figure
    plt.close()  # Close the plot to free memory

# Save Pearson heatmap
plot_heatmap(corr_matrix_pearson, 'Pearson Correlation Heatmap', 'pearson_heatmap.png')

# Save Spearman heatmap
plot_heatmap(corr_matrix_spearman, 'Spearman Correlation Heatmap', 'spearman_heatmap.png')

In [7]:
# Compute the difference between Pearson and Spearman correlations
corr_diff = corr_matrix_pearson - corr_matrix_spearman

# Plot and save the heatmap of the differences
plt.figure(figsize=(15, 10))
sns.heatmap(corr_diff, cmap='coolwarm', center=0, square=True)
plt.title('Difference Between Pearson and Spearman Correlations')
plt.xticks(ticks=range(len(corr_diff.columns)), labels=corr_diff.columns, fontsize=7, rotation=90)
plt.yticks(ticks=range(len(corr_diff.index)), labels=corr_diff.index, fontsize=7, rotation=0)
plt.savefig('correlation_difference_heatmap.png', dpi=300, bbox_inches='tight')  # Save the figure
plt.close()

In [8]:
# Define dependent variable(s)
dependent_vars = ['A', 'LA', 'PA']  # Replace with actual dependent variable(s)

# Get only independent variables
independent_vars = [col for col in df_att.columns if col not in dependent_vars + ['H', 'V']]

# Compute Pearson and Spearman correlations for only the dependent variable(s)
corr_pearson_dep = df_att.drop(columns=['H', 'V']).corr(method='pearson').loc[dependent_vars, independent_vars]
corr_spearman_dep = df_att.drop(columns=['H', 'V']).corr(method='spearman').loc[dependent_vars, independent_vars]

# Function to plot heatmap for dependent variables
def plot_heatmap_dep(corr_matrix, title, save_path):
    plt.figure(figsize=(10, len(dependent_vars)))  # Adjust height dynamically
    sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, cbar=True)
    plt.title(title)
    plt.xticks(ticks=range(len(corr_matrix.columns)), labels=corr_matrix.columns, fontsize=8, rotation=90)
    plt.yticks(ticks=range(len(corr_matrix.index)), labels=corr_matrix.index, fontsize=8, rotation=0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

# Save heatmaps for dependent variables
plot_heatmap_dep(corr_pearson_dep, 'Pearson Correlation (Dependent Variables)', 'pearson_dep_heatmap.png')
plot_heatmap_dep(corr_spearman_dep, 'Spearman Correlation (Dependent Variables)', 'spearman_dep_heatmap.png')

In [9]:
##### Compute the difference for only the dependent variable(s)
corr_diff_dep = corr_pearson_dep - corr_spearman_dep

# Plot and save heatmap for the difference
plt.figure(figsize=(10, len(dependent_vars)))  # Adjust height dynamically
sns.heatmap(corr_diff_dep, cmap='coolwarm', center=0, cbar=True)
plt.title('Difference Between Pearson and Spearman Correlations (Dependent Variables)')
plt.xticks(ticks=range(len(corr_diff_dep.columns)), labels=corr_diff_dep.columns, fontsize=8, rotation=90)
plt.yticks(ticks=range(len(corr_diff_dep.index)), labels=corr_diff_dep.index, fontsize=8, rotation=0)
plt.savefig('correlation_difference_dep_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()

In [10]:
# Suppress RuntimeWarnings in this block
warnings.filterwarnings("ignore", category=RuntimeWarning)

# variance inflation factor
# Select numerical columns (excluding target variable)
df_numeric = df_att.drop(columns=['H', 'V', 'A', 'LA', 'PA'])  # Exclude categorical variables

# Add a constant for intercept
X = df_numeric.copy()
X['Intercept'] = 1  # Required for VIF calculation

# Compute VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Drop the intercept row for interpretation
vif_data = vif_data[vif_data["Feature"] != "Intercept"]

# Re-enable warnings after this block
warnings.filterwarnings("default", category=RuntimeWarning)

# Display results
vif_data[vif_data['VIF'] >= 10]

,Feature,VIF
0,HRk,16.490040
2,HW,inf
3,HL,inf
4,HOL,inf
5,HPTS%,68.188705
6,HSOW,13.459593
7,HSOL,14.435630
8,HSRS,4013.965945
9,HSOS,28.413536
10,HGF/G,2010.902521


In [11]:
# VIF FEATURE SELECTION

# Suppress RuntimeWarnings in this block
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Function to compute VIF and iteratively remove high VIF features
def calculate_vif(df, threshold=10):
    X = df.copy()
    X['Intercept'] = 1  # Required for VIF calculation
    
    while True:
        # Compute VIF for each feature
        vif_data = pd.DataFrame()
        vif_data["Feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        
        # Drop intercept row
        vif_data = vif_data[vif_data["Feature"] != "Intercept"]
        
        # Find the feature with the highest VIF
        max_vif = vif_data["VIF"].max()
        if max_vif < threshold:
            break  # Stop if all VIF values are below the threshold
        
        # Identify the feature to remove
        feature_to_remove = vif_data.loc[vif_data["VIF"].idxmax(), "Feature"]
        print(f"Removing {feature_to_remove} with VIF {max_vif:.2f}")
        
        # Drop the feature with the highest VIF
        X = X.drop(columns=[feature_to_remove])
    
    return X.drop(columns=['Intercept'])  # Return dataframe without high-VIF features

# Run the VIF reduction process (CAP --> drop LCAP)
df_numeric_cap = df_cap.drop(columns=['H', 'V', 'A', 'PA'])
df_numeric_lcap = df_lcap.drop(columns=['H', 'V', 'LA'])

df_reduced_cap = calculate_vif(df_numeric_cap)

# Run the VIF reduction process (LCAP --> drop CAP)
df_reduced_lcap = calculate_vif(df_numeric_lcap)

print(df_reduced_cap.columns)
print(df_reduced_lcap.columns)

selected_feats_cap_vif = df_reduced_cap.columns
selected_feats_lcap_vif = df_reduced_lcap.columns

# Re-enable warnings after this block
warnings.filterwarnings("default", category=RuntimeWarning)

Removing HW with VIF inf
Removing MDAY with VIF inf
Removing VSRS with VIF 4049.78
Removing HSRS with VIF 4007.60
Removing VPPA with VIF 79.58
Removing VPTS% with VIF 63.58
Removing HPTS% with VIF 62.99
Removing VW with VIF 32.97
Removing VL with VIF 22.49
Removing VPPO with VIF 19.32
Removing HPPO with VIF 18.06
Removing HL with VIF 17.27
Removing VGA/G with VIF 10.36
Removing HGA/G with VIF 10.28
Removing HW with VIF inf
Removing MDAY with VIF inf
Removing VSRS with VIF 4049.76
Removing HSRS with VIF 4007.73
Removing VPPA with VIF 79.58
Removing VPTS% with VIF 63.58
Removing HPTS% with VIF 63.01
Removing VW with VIF 32.96
Removing VL with VIF 22.49
Removing VPPO with VIF 19.32
Removing HPPO with VIF 18.05
Removing HL with VIF 17.27
Removing VGA/G with VIF 10.36
Removing HGA/G with VIF 10.28
Index(['HRk', 'HAvAge', 'HOL', 'HSOW', 'HSOL', 'HSOS', 'HGF/G', 'HPP', 'HPPA',
       'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO',
       'HCAN', 'VRk', 'VAvAge', 'VOL', 'VSO

In [12]:
# LASSO stability selection
def stability_selection_lasso(dep, X, y, alpha_range=np.logspace(-4, 1, 10), 
                              n_resampling=100, selection_threshold=0.5, name_feats=''):
    """
    Performs LASSO-based stability selection, finds the best alpha, and reports RMSE/MAE/R² for train/test (averaged over all resampling iterations).
    
    Parameters:
        dep (str): Dependent variable type ('A', 'LA', or 'PA').
        X (DataFrame): Feature matrix.
        y (Series): Target variable.
        alpha_range (array-like): List of alpha values to test.
        n_resampling (int): Number of resampling iterations.
        selection_threshold (float): Minimum fraction of resamples a feature must be selected in.

    Returns:
        best_alpha (float): Optimal alpha with lowest RMSE.
        selected_features (list): Names of selected features using best_alpha.
        stats (dict): Average RMSE, MAE, R² for train/test.
    """
    # Split into train/test sets (y is not scaled)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Standardize X only
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    results = []
    warning_alphas = []
    model_stats = {}

    for alpha in alpha_range:
        selection_counts = np.zeros(X.shape[1])
        
        # Initialize statistics to accumulate over iterations
        rmse_train_all = []
        rmse_test_all = []
        mae_train_all = []
        mae_test_all = []
        r2_train_all = []
        r2_test_all = []

        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always", ConvergenceWarning)
            
            for _ in range(n_resampling):
                # bootstrap resampling
                X_sample, y_sample = resample(X_train_scaled, y_train, n_samples=int(0.7 * len(y_train)), random_state=None)
                model = Lasso(alpha=alpha)
                model.fit(X_sample, y_sample)
                selection_counts += (model.coef_ != 0)

                # Make predictions
                y_train_pred = model.predict(X_train_scaled)
                y_test_pred = model.predict(X_test_scaled)

                # Compute statistics for this resample
                stats = compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred)
                
                # Collect stats for averaging
                rmse_train_all.append(stats["RMSE Train"])
                rmse_test_all.append(stats["RMSE Test"])
                mae_train_all.append(stats["MAE Train"])
                mae_test_all.append(stats["MAE Test"])
                r2_train_all.append(stats["R² Train"])
                r2_test_all.append(stats["R² Test"])

            # Average the statistics over all resampling iterations
            avg_rmse_train = np.mean(rmse_train_all)
            avg_rmse_test = np.mean(rmse_test_all)
            avg_mae_train = np.mean(mae_train_all)
            avg_mae_test = np.mean(mae_test_all)
            avg_r2_train = np.mean(r2_train_all)
            avg_r2_test = np.mean(r2_test_all)

            # Store the averaged stats for this alpha
            model_stats[alpha] = {
                "RMSE Train": avg_rmse_train,
                "RMSE Test": avg_rmse_test,
                "MAE Train": avg_mae_train,
                "MAE Test": avg_mae_test,
                "R² Train": avg_r2_train,
                "R² Test": avg_r2_test
            }

        # Compute feature selection stability
        selection_frequencies = selection_counts / n_resampling
        selected_features = np.where(selection_frequencies >= selection_threshold)[0]
        num_selected = len(selected_features)

        results.append((alpha, num_selected, selected_features))

    # Extract alphas and feature counts
    alphas, feature_counts, feature_indices_list = zip(*results)

    # Plot feature count vs. alpha
    plt.figure(figsize=(12, 6))  # Increased figure size for better spacing
    plt.subplot(1, 2, 1)
    plt.plot(alphas, feature_counts, marker='o', linestyle='-', color='red')
    plt.xscale('log')
    plt.xlabel('Alpha')
    plt.ylabel('Number of Features Selected')
    plt.title('Number of Features Selected for Different Alpha Values')
    
    # Plot RMSE vs. Alpha (use averaged RMSE Test)
    plt.subplot(1, 2, 2)
    plt.plot(model_stats.keys(), [stat["RMSE Test"] for stat in model_stats.values()], marker='o', linestyle='-', color='red')
    plt.xscale('log')
    plt.xlabel('Alpha')
    plt.ylabel('RMSE (Test)')
    plt.title('Test RMSE for Different Alpha Values')
    
    # Adjust layout for spacing between subplots
    plt.subplots_adjust(wspace=0.3, hspace=0.2)  # Increase horizontal and vertical space
    
    # Save the plot with a dynamic filename
    plt.savefig(f"LASSO_stability_{dep}_{name_feats}.png", dpi=300, bbox_inches='tight')
    
    # Close the plot to free up memory and avoid duplicate plots
    plt.close()

    best_alpha = min(model_stats, key=lambda a: model_stats[a]["RMSE Test"])

    # Recompute selection counts for best alpha
    selection_counts_best = np.zeros(X.shape[1])
    for _ in range(n_resampling):
        X_sample, y_sample = resample(X_train_scaled, y_train, n_samples=int(0.7 * len(y_train)), random_state=None)
        model = Lasso(alpha=best_alpha)
        model.fit(X_sample, y_sample)
        selection_counts_best += (model.coef_ != 0)

    selection_frequencies_best = selection_counts_best / n_resampling
    selected_feature_indices = np.where(selection_frequencies_best >= selection_threshold)[0]
    selected_feature_names = X.columns[selected_feature_indices]

    # --- Plot bar chart ---
    plt.figure(figsize=(12, 6))
    plt.bar(X.columns, selection_frequencies_best, color="red")
    plt.axhline(selection_threshold, color="black", linestyle="--", label="Selection Threshold")
    plt.xticks(rotation=90)
    plt.xlabel("Feature")
    plt.ylabel("Selection Frequency")
    plt.title(f"LASSO Feature Stability for {dep} (α = {best_alpha:.4g})")
    plt.legend()
    plt.savefig(f"LASSO_stability_barchart_{dep}_{name_feats}.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # Convert indices to feature names
    selected_feature_names = X.columns[selected_feature_indices]

    print(f"Optimal alpha: {best_alpha}")
    print(f"Number of features selected: {len(selected_feature_names)}")
    print("Selected Features:", selected_feature_names.tolist())

    # Print statistics for best alpha
    best_stats = model_stats[best_alpha]
    print("\nPerformance Metrics (Train/Test):")
    for metric, value in best_stats.items():
        print(f"{metric}: {value:.4f}")

    # Print alphas that triggered convergence warnings
    if warning_alphas:
        print(f"\n⚠️ Convergence warnings occurred for alpha values: {warning_alphas}")

    return best_alpha, selected_feature_names, best_stats

In [13]:
# all features
# tends to not converge because of high degree of multicollinearity

print('Dependent variable: A')
alpha_cap_a, selected_feat_cap_a_lasso, best_stats_a = stability_selection_lasso('A', X_cap_a, y_cap_a, 
                                                                                 alpha_range = np.logspace(-1, 1, 10), 
                                                                                 name_feats = 'full_list')

Dependent variable: A
Optimal alpha: 3.593813663804626
Number of features selected: 50
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VL', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1287.6284
RMSE Test: 1311.4428
MAE Train: 918.3517
MAE Test: 917.6337
R² Train: 0.5477
R² Test: 0.5249


In [14]:
# all featuress
print('Dependent variable: PA')
alpha_cap_pa, selected_feat_cap_pa_lasso, best_stats_pa = stability_selection_lasso('PA', X_cap_pa, y_cap_pa, 
                                                                                    alpha_range = np.logspace(-5, -3, 10),
                                                                                    name_feats = 'full_list')

Dependent variable: PA
Optimal alpha: 0.0001291549665014884
Number of features selected: 52
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VL', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VPPOA', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1287.5812
RMSE Test: 1311.5812
MAE Train: 922.0697
MAE Test: 921.2737
R² Train: 0.5477
R² Test: 0.5248


In [15]:
# all features
print('Dependent variable: LA')
alpha_lcap, selected_feat_lcap_lasso, best_stats_la = stability_selection_lasso('LA', X_lcap, y_lcap, 
                                                                                alpha_range = np.logspace(-4, -3, 10),
                                                                                name_feats = 'full_list')

Dependent variable: LA
Optimal alpha: 0.0002782559402207126
Number of features selected: 47
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VPP', 'VPPA', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'LCAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1291.9437
RMSE Test: 1314.4763
MAE Train: 941.4653
MAE Test: 940.6836
R² Train: 0.5446
R² Test: 0.5227


In [16]:
# vif features
print('Dependent variable: A')
alpha_cap_a_vif, selected_feat_cap_a_lasso_vif, best_stats_a_vif = stability_selection_lasso('A', df_reduced_cap, y_cap_a, 
                                                                                 alpha_range = np.logspace(-1, 2, 10),
                                                                                 name_feats = 'vif')

print('\nDependent variable: PA')
alpha_cap_pa_vif, selected_feat_cap_pa_lasso_vif, best_stats_pa_vif = stability_selection_lasso('PA', df_reduced_cap, y_cap_pa, 
                                                                                    alpha_range = np.logspace(-5, -2, 10),
                                                                                    name_feats = 'vif')

print('\nDependent variable: LA')
alpha_lcap_vif, selected_feat_lcap_lasso_vif, best_stats_la_vif = stability_selection_lasso('LA', df_reduced_lcap, y_lcap, 
                                                                                alpha_range = np.logspace(-5, -2, 10),
                                                                                name_feats = 'vif')

Dependent variable: A
Optimal alpha: 10.0
Number of features selected: 36
Selected Features: ['HRk', 'HAvAge', 'HOL', 'HSOW', 'HSOL', 'HSOS', 'HGF/G', 'HSH', 'HSHA', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VOL', 'VSOW', 'VSOS', 'VGF/G', 'VPPOA', 'VSH', 'VSHA', 'VoPIM/G', 'VS%', 'VSV%', 'VCAN', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1310.5400
RMSE Test: 1329.9340
MAE Train: 936.6449
MAE Test: 937.3076
R² Train: 0.5314
R² Test: 0.5114

Dependent variable: PA
Optimal alpha: 0.001
Number of features selected: 25
Selected Features: ['HRk', 'HAvAge', 'HSOW', 'HSOS', 'HSH', 'HSHA', 'HoPIM/G', 'HS%', 'HSO', 'HCAN', 'VRk', 'VPPOA', 'VSH', 'VSHA', 'VoPIM/G', 'VSV%', 'VCAN', 'TDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1313.5740
RMSE Test: 1329.8223
MAE Train: 939.8500
MAE Test: 939.1487
R² Train: 0.5292
R² Test: 0.5115

Dependent v

In [17]:
def stability_selection_rf(dep, X, y, n_estimators=100, n_resampling=100, selection_threshold=0.5, importance_threshold=0.5):
    """
    Performs stability selection for feature importance using Random Forest.
    
    Parameters:
        dep (str): Dependent variable type ('A', 'LA', or 'PA').
        X (DataFrame): Feature matrix.
        y (Series): Target variable.
        n_estimators (int): Number of trees in Random Forest.
        n_resampling (int): Number of resampling iterations.
        selection_threshold (float): Minimum fraction of resamples a feature must be important in.
        importance_threshold (float): Fraction of most important features to consider in each iteration.

    Returns:
        selected_features (list): Names of selected features.
        stats (dict): Average RMSE, MAE, R² for train/test.
    """

    # Split into train/test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Track feature importance frequency
    feature_importance_counts = np.zeros(X.shape[1])

    # Initialize statistics tracking
    rmse_train_all, rmse_test_all = [], []
    mae_train_all, mae_test_all = [], []
    r2_train_all, r2_test_all = [], []

    for _ in range(n_resampling):
        # Bootstrap resample
        X_sample, y_sample = resample(X_train, y_train, n_samples=int(0.7 * len(y_train)), random_state=None)

        # Train Random Forest
        rf = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
        rf.fit(X_sample, y_sample)

        # Determine number of features to consider based on importance_threshold
        num_top_features = max(1, int(importance_threshold * len(X.columns)))  # Ensure at least 1 feature is selected
        importance_ranking = np.argsort(rf.feature_importances_)[::-1]  # Indices of features sorted by importance
        top_features = importance_ranking[:num_top_features]  
        feature_importance_counts[top_features] += 1

        # Make predictions
        y_train_pred = rf.predict(X_train)
        y_test_pred = rf.predict(X_test)

        # Compute statistics for this resample
        stats = compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred)
        
        # Collect stats for averaging
        rmse_train_all.append(stats["RMSE Train"])
        rmse_test_all.append(stats["RMSE Test"])
        mae_train_all.append(stats["MAE Train"])
        mae_test_all.append(stats["MAE Test"])
        r2_train_all.append(stats["R² Train"])
        r2_test_all.append(stats["R² Test"])

    # Compute stability selection scores
    selection_frequencies = feature_importance_counts / n_resampling
    selected_features_indices = np.where(selection_frequencies >= selection_threshold)[0]
    selected_feature_names = X.columns[selected_features_indices]

    # Compute averaged statistics
    stats = {
        "RMSE Train": np.mean(rmse_train_all),
        "RMSE Test": np.mean(rmse_test_all),
        "MAE Train": np.mean(mae_train_all),
        "MAE Test": np.mean(mae_test_all),
        "R² Train": np.mean(r2_train_all),
        "R² Test": np.mean(r2_test_all),
    }

    # Plot feature stability selection (VERTICAL bar chart)
    plt.figure(figsize=(12, 6))
    plt.bar(X.columns, selection_frequencies, color="red")
    plt.axhline(selection_threshold, color="black", linestyle="--", label="Selection Threshold")
    
    plt.xticks(rotation=90)  # Rotate x-axis labels for readability
    plt.xlabel("Feature")
    plt.ylabel("Selection Frequency")
    plt.title(f"Feature Selection Stability for {dep}")
    plt.legend()

    plt.savefig(f"RF_stability_{dep}.png", dpi=300, bbox_inches='tight')

    # Close the plot to free up memory and avoid duplicate plots
    plt.close()

    print(f"Number of selected features: {len(selected_feature_names)}")
    print("Selected Features:", selected_feature_names.tolist())

    print("\nPerformance Metrics (Train/Test):")
    for metric, value in stats.items():
        print(f"{metric}: {value:.4f}")

    return selected_feature_names, stats

In [18]:
print('Dependent variable: A')
selected_feat_cap_a_rf, best_stats_a_rf = stability_selection_rf('A', X_cap_a, y_cap_a)

Dependent variable: A
Number of selected features: 30
Selected Features: ['HRk', 'HAvAge', 'HW', 'HPTS%', 'HSRS', 'HSOS', 'HGF/G', 'HGA/G', 'HPP', 'HPPA', 'HPPOA', 'HSH', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'VAvAge', 'VGF/G', 'VGA/G', 'VPP', 'VPPO', 'VPPOA', 'VPK%', 'VSH', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'SDAY', 'CAP']

Performance Metrics (Train/Test):
RMSE Train: 686.0990
RMSE Test: 925.1066
MAE Train: 360.5590
MAE Test: 545.6120
R² Train: 0.8716
R² Test: 0.7635


In [19]:
print('Dependent variable: PA')
selected_feat_cap_pa_rf, best_stats_pa_rf = stability_selection_rf('PA', X_cap_pa, y_cap_pa)

Dependent variable: PA
Number of selected features: 28
Selected Features: ['HRk', 'HAvAge', 'HOL', 'HSRS', 'HSOS', 'HGF/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HPIM/G', 'HoPIM/G', 'HSV%', 'VAvAge', 'VGF/G', 'VGA/G', 'VPP', 'VPPO', 'VPPOA', 'VPK%', 'VSH', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'SDAY', 'CAP']

Performance Metrics (Train/Test):
RMSE Train: 685.8186
RMSE Test: 922.8333
MAE Train: 362.6396
MAE Test: 547.4267
R² Train: 0.8717
R² Test: 0.7647


In [20]:
print('Dependent variable: LA')
selected_feat_lcap_rf, best_stats_la_rf = stability_selection_rf('LA', X_lcap, y_lcap)

Dependent variable: LA
Number of selected features: 31
Selected Features: ['HRk', 'HAvAge', 'HW', 'HPTS%', 'HSRS', 'HSOS', 'HGF/G', 'HGA/G', 'HPP', 'HPPA', 'HPPOA', 'HSH', 'HPIM/G', 'HoPIM/G', 'HS%', 'VAvAge', 'VSOS', 'VGF/G', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VPPOA', 'VPK%', 'VSH', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'SDAY', 'LCAP']

Performance Metrics (Train/Test):
RMSE Train: 687.0504
RMSE Test: 927.1426
MAE Train: 362.4872
MAE Test: 549.3294
R² Train: 0.8712
R² Test: 0.7625


In [21]:
#### OLS WITH LASSO SELECTED FEATS

# A

# Subset the data to include only the selected features
X_selected = X_cap_a[selected_feat_cap_a_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_a, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      A   R-squared:                       0.548
Model:                            OLS   Adj. R-squared:                  0.546
Method:                 Least Squares   F-statistic:                     249.2
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        17:53:49   Log-Likelihood:                -88583.
No. Observations:               10327   AIC:                         1.773e+05
Df Residuals:                   10276   BIC:                         1.776e+05
Df Model:                          50                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.246e+04   3568.496      3.491      0.000    5461.369    1.95e+04
HRk          -56.8338      5.289    -10.745      0.000     -67.202     -46.466
HAvAge        81.8940     13.599      6.022      0.000      55.238     108.550
HW           -23.8908      6.654     -3.590      0.000     -36.935     -10.847
HL            76.0849      7.021     10.836      0.000      62.322      89.848
HOL           16.5328      8.142      2.031      0.042       0.574      32.492
HSOW          56.5920      7.249      7.807      0.000      42.383      70.801
HSOL          15.8763      8.804      1.803      0.071      -1.380      33.133
HSRS         918.7731    119.526      7.687      0.000     684.480    1153.067
HSOS         219.2880    397.926      0.551      0.582    -560.725     999.301
HGA/G      -1161.4239    120.693     -9.623      0.000   -1398.005    -924.843
HPP            7.2252      2.431      2.972      0.003       2.460      11.990
HPPO          -5.4666      0.977     -5.594      0.000      -7.382      -3.551
HPPA          10.7263      2.410      4.451      0.000       6.002      15.451
HPPOA         -4.0072      0.949     -4.223      0.000      -5.867      -2.147
HSH          -20.8946      4.943     -4.227      0.000     -30.584     -11.205
HSHA          25.4224      5.684      4.473      0.000      14.281      36.563
HPIM/G        71.9691     12.138      5.929      0.000      48.175      95.763
HS%          155.4432     25.332      6.136      0.000     105.787     205.099
HSV%       -1.947e+04   2577.131     -7.554      0.000   -2.45e+04   -1.44e+04
HSO          -23.9996      7.314     -3.281      0.001     -38.337      -9.662
HCAN         183.4109     32.703      5.608      0.000     119.307     247.515
VRk          -21.6509      5.179     -4.181      0.000     -31.802     -11.499
VAvAge        -0.5503     13.328     -0.041      0.967     -26.675      25.574
VL            21.4005      6.699      3.195      0.001       8.270      34.531
VOL           13.2089      7.226      1.828      0.068      -0.955      27.373
VSOW          -8.9568      7.073     -1.266      0.205     -22.821       4.907
VSOL          -8.9378      8.378     -1.067      0.286     -25.361       7.485
VSOS         457.6339    395.013      1.159      0.247    -316.668    1231.936
VGA/G       -138.1127    104.300     -1.324      0.185    -342.560      66.335
VPP            1.1470      2.343      0.490      0.624      -3.445       5.739
VPPO           0.1665      1.195      0.139      0.889      -2.177       2.509
VPPA           3.3738      2.190      1.541      0.123      -0.919       7.667
VSH            6.9380      4.827      1.437      0.151      -2.523      16.399
VSHA           7.1668      5.687      1.260      0.208      -3.980      18.314
VPIM/G        -3.5897     19.164     -0.187      0.851     -41.155      33.975
Vo

In [22]:
# PA

# Subset the data to include only the selected features
X_selected = X_cap_pa[selected_feat_cap_pa_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_pa, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     PA   R-squared:                       0.171
Model:                            OLS   Adj. R-squared:                  0.167
Method:                 Least Squares   F-statistic:                     40.80
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        17:53:49   Log-Likelihood:                 12649.
No. Observations:               10327   AIC:                        -2.519e+04
Df Residuals:                   10274   BIC:                        -2.481e+04
Df Model:                          52                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7324      0.198      8.753      0.000       1.344       2.120
HRk           -0.0030      0.000    -10.130      0.000      -0.004      -0.002
HAvAge         0.0040      0.001      5.296      0.000       0.003       0.005
HW            -0.0014      0.000     -3.776      0.000      -0.002      -0.001
HL             0.0041      0.000     10.478      0.000       0.003       0.005
HOL            0.0010      0.000      2.137      0.033    7.99e-05       0.002
HSOW           0.0034      0.000      8.466      0.000       0.003       0.004
HSOL           0.0008      0.000      1.685      0.092      -0.000       0.002
HSRS           0.0557      0.007      8.386      0.000       0.043       0.069
HSOS          -0.0069      0.022     -0.312      0.755      -0.050       0.036
HGA/G         -0.0611      0.007     -9.069      0.000      -0.074      -0.048
HPP            0.0004      0.000      2.851      0.004       0.000       0.001
HPPO          -0.0002   7.59e-05     -2.155      0.031      -0.000   -1.48e-05
HPPA           0.0006      0.000      4.515      0.000       0.000       0.001
HPPOA         -0.0003    6.7e-05     -4.936      0.000      -0.000      -0.000
HSH           -0.0011      0.000     -3.954      0.000      -0.002      -0.001
HSHA           0.0012      0.000      3.785      0.000       0.001       0.002
HPIM/G         0.0081      0.002      4.842      0.000       0.005       0.011
HoPIM/G       -0.0042      0.002     -2.389      0.017      -0.008      -0.001
HS%            0.0079      0.001      5.605      0.000       0.005       0.011
HSV%          -1.1598      0.143     -8.131      0.000      -1.439      -0.880
HSO           -0.0012      0.000     -2.986      0.003      -0.002      -0.000
HCAN           0.0090      0.002      4.940      0.000       0.005       0.013
VRk           -0.0013      0.000     -4.353      0.000      -0.002      -0.001
VAvAge      5.591e-06      0.001      0.008      0.994      -0.001       0.001
VL             0.0012      0.000      3.295      0.001       0.001       0.002
VOL            0.0008      0.000      2.021      0.043    2.43e-05       0.002
VSOW          -0.0006      0.000     -1.453      0.146      -0.001       0.000
VSOL          -0.0006      0.000     -1.286      0.198      -0.002       0.000
VSOS           0.0256      0.022      1.168      0.243      -0.017       0.069
VGA/G         -0.0073      0.006     -1.248      0.212      -0.019       0.004
VPP         8.203e-05      0.000      0.626      0.531      -0.000       0.000
VPPO        4.019e-05   7.65e-05      0.525      0.599      -0.000       0.000
VPPA           0.0002      0.000      1.651      0.099   -4.01e-05       0.000
VPPOA       -5.35e-05   6.73e-05     -0.794      0.427      -0.000    7.85e-05
VSH            0.0004      0.000      1.657      0.098    -8.1e-05       0.001
VS

In [23]:
# LA

# Subset the data to include only the selected features
X_selected = X_lcap[selected_feat_lcap_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_lcap, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     LA   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.519
Method:                 Least Squares   F-statistic:                     238.3
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        17:53:49   Log-Likelihood:                 11586.
No. Observations:               10327   AIC:                        -2.308e+04
Df Residuals:                   10279   BIC:                        -2.273e+04
Df Model:                          47                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0402      0.226     -0.178      0.859      -0.484       0.403
HRk           -0.0031      0.000     -9.620      0.000      -0.004      -0.002
HAvAge         0.0045      0.001      5.424      0.000       0.003       0.006
HW            -0.0007      0.000     -2.279      0.023      -0.001      -0.000
HL             0.0050      0.000     13.802      0.000       0.004       0.006
HOL            0.0017      0.000      3.961      0.000       0.001       0.003
HSOW           0.0034      0.000      7.720      0.000       0.003       0.004
HSOL           0.0009      0.001      1.605      0.109      -0.000       0.002
HSRS           0.0560      0.007      7.757      0.000       0.042       0.070
HSOS          -0.0154      0.024     -0.631      0.528      -0.063       0.032
HGA/G         -0.0690      0.007     -9.598      0.000      -0.083      -0.055
HPP            0.0004      0.000      2.871      0.004       0.000       0.001
HPPO          -0.0003   5.93e-05     -5.535      0.000      -0.000      -0.000
HPPA           0.0007      0.000      4.576      0.000       0.000       0.001
HPPOA         -0.0002    5.8e-05     -4.067      0.000      -0.000      -0.000
HSH           -0.0011      0.000     -3.734      0.000      -0.002      -0.001
HSHA           0.0014      0.000      3.939      0.000       0.001       0.002
HPIM/G         0.0051      0.001      6.830      0.000       0.004       0.007
HS%            0.0088      0.002      5.666      0.000       0.006       0.012
HSV%          -1.2921      0.158     -8.191      0.000      -1.601      -0.983
HSO           -0.0012      0.000     -2.763      0.006      -0.002      -0.000
HCAN           0.0122      0.002      6.128      0.000       0.008       0.016
VRk           -0.0006      0.000     -3.263      0.001      -0.001      -0.000
VAvAge        -0.0002      0.001     -0.262      0.793      -0.002       0.001
VOL            0.0002      0.000      0.396      0.692      -0.001       0.001
VSOW          -0.0008      0.000     -1.834      0.067      -0.002    5.39e-05
VSOL          -0.0006      0.001     -1.253      0.210      -0.002       0.000
VSOS           0.0177      0.024      0.750      0.453      -0.029       0.064
VPP        -3.342e-05      0.000     -0.272      0.786      -0.000       0.000
VPPA           0.0002      0.000      1.885      0.060   -9.13e-06       0.000
VSH            0.0003      0.000      1.179      0.238      -0.000       0.001
VSHA           0.0006      0.000      1.793      0.073   -5.55e-05       0.001
VPIM/G        -0.0005      0.001     -0.461      0.645      -0.003       0.002
VoPIM/G        0.0018      0.001      1.549      0.121      -0.000       0.004
VS%           -0.0020      0.001     -1.624      0.104      -0.004       0.000
VSV%           0.2189      0.128      1.706      0.088      -0.033       0.471
VS

In [24]:
#### OLS WITH RF SELECTED FEATS

# A

# Subset the data to include only the selected features
X_selected = X_cap_a[selected_feat_cap_a_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_a, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      A   R-squared:                       0.533
Model:                            OLS   Adj. R-squared:                  0.531
Method:                 Least Squares   F-statistic:                     391.0
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        17:53:50   Log-Likelihood:                -88757.
No. Observations:               10327   AIC:                         1.776e+05
Df Residuals:                   10296   BIC:                         1.778e+05
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.207e+04   3658.253      3.298      0.001    4894.343    1.92e+04
HRk          -46.1597      5.732     -8.053      0.000     -57.395     -34.924
HAvAge        58.3549     13.644      4.277      0.000      31.611      85.099
HW            -7.7647      5.688     -1.365      0.172     -18.914       3.384
HPTS%      -5395.8097    775.823     -6.955      0.000   -6916.573   -3875.047
HSRS        2843.5811    404.673      7.027      0.000    2050.343    3636.819
HSOS       -3253.1085    540.760     -6.016      0.000   -4313.103   -2193.114
HGF/G      -2187.3048    391.716     -5.584      0.000   -2955.145   -1419.465
HGA/G       1087.8067    381.524      2.851      0.004     339.945    1835.668
HPP            2.2534      2.262      0.996      0.319      -2.181       6.688
HPPA          14.0444      2.431      5.776      0.000       9.279      18.810
HPPOA         -2.8488      0.991     -2.876      0.004      -4.790      -0.907
HSH          -14.3574      4.961     -2.894      0.004     -24.082      -4.633
HPIM/G        75.7581     22.714      3.335      0.001      31.235     120.281
HoPIM/G      -37.3914     21.627     -1.729      0.084     -79.785       5.003
HS%          172.4717     25.154      6.857      0.000     123.166     221.778
HSV%       -1.613e+04   2583.708     -6.243      0.000   -2.12e+04   -1.11e+04
VAvAge        -3.3896     13.290     -0.255      0.799     -29.441      22.662
VGF/G        -26.3938     78.997     -0.334      0.738    -181.243     128.455
VGA/G        -79.8153     76.605     -1.042      0.297    -229.975      70.345
VPP           -0.7880      2.298     -0.343      0.732      -5.293       3.717
VPPO           3.8624      1.297      2.979      0.003       1.321       6.404
VPPOA          1.8700      1.122      1.667      0.096      -0.329       4.069
VPK%          -5.5549      5.490     -1.012      0.312     -16.315       5.206
VSH            7.6499      4.872      1.570      0.116      -1.900      17.200
VPIM/G        -5.1084     29.994     -0.170      0.865     -63.902      53.686
VoPIM/G       -6.5711     31.157     -0.211      0.833     -67.645      54.503
VS%           10.5022     25.370      0.414      0.679     -39.228      60.232
VSV%        4047.4250   2566.549      1.577      0.115    -983.510    9078.360
SDAY         313.8676     30.134     10.416      0.000     254.798     372.937
CAP            1.0714      0.011     96.876      0.000       1.050       1.093
==============================================================================
Omnibus:                     2240.157   Durbin-Watson:                   1.974
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             4723.508
Skew:                          -1.270   Prob(JB):                         0.00
Kurtosis:                       5.128   Cond. No.                     6.28e+06
==

In [25]:
#### OLS WITH RF SELECTED FEATS

# PA

# Subset the data to include only the selected features
X_selected = X_cap_pa[selected_feat_cap_pa_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_pa, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     PA   R-squared:                       0.135
Model:                            OLS   Adj. R-squared:                  0.133
Method:                 Least Squares   F-statistic:                     57.38
Date:                Fri, 04 Apr 2025   Prob (F-statistic):          9.36e-298
Time:                        17:53:50   Log-Likelihood:                 12428.
No. Observations:               10327   AIC:                        -2.480e+04
Df Residuals:                   10298   BIC:                        -2.459e+04
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.5433      0.201      7.676      0.000       1.149       1.937
HRk           -0.0008      0.000     -3.430      0.001      -0.001      -0.000
HAvAge         0.0021      0.001      2.766      0.006       0.001       0.004
HOL           -0.0004      0.000     -1.282      0.200      -0.001       0.000
HSRS           0.0620      0.006     10.846      0.000       0.051       0.073
HSOS          -0.1098      0.022     -5.055      0.000      -0.152      -0.067
HGF/G         -0.0366      0.005     -6.771      0.000      -0.047      -0.026
HPP            0.0001      0.000      1.137      0.256      -0.000       0.000
HPPO       -9.568e-05   6.93e-05     -1.381      0.167      -0.000    4.01e-05
HPPA           0.0007      0.000      5.376      0.000       0.000       0.001
HPPOA         -0.0001   6.61e-05     -2.099      0.036      -0.000   -9.18e-06
HSH           -0.0007      0.000     -2.383      0.017      -0.001      -0.000
HPIM/G         0.0033      0.002      1.985      0.047    4.02e-05       0.007
HoPIM/G       -0.0003      0.002     -0.176      0.860      -0.004       0.003
HSV%          -0.8981      0.144     -6.253      0.000      -1.180      -0.617
VAvAge        -0.0002      0.001     -0.258      0.797      -0.002       0.001
VGF/G         -0.0024      0.004     -0.558      0.577      -0.011       0.006
VGA/G         -0.0062      0.004     -1.468      0.142      -0.015       0.002
VPP        -5.552e-05      0.000     -0.435      0.664      -0.000       0.000
VPPO           0.0002   7.04e-05      3.025      0.002     7.5e-05       0.000
VPPOA        9.77e-05   6.19e-05      1.577      0.115   -2.37e-05       0.000
VPK%          -0.0003      0.000     -1.023      0.306      -0.001       0.000
VSH            0.0004      0.000      1.634      0.102   -8.81e-05       0.001
VPIM/G        -0.0002      0.002     -0.121      0.903      -0.003       0.003
VoPIM/G       -0.0003      0.002     -0.173      0.863      -0.004       0.003
VS%            0.0007      0.001      0.512      0.608      -0.002       0.003
VSV%           0.1824      0.142      1.282      0.200      -0.097       0.461
SDAY           0.0175      0.002     10.429      0.000       0.014       0.021
CAP         5.541e-06   6.13e-07      9.045      0.000    4.34e-06    6.74e-06
==============================================================================
Omnibus:                     2155.946   Durbin-Watson:                   1.982
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             4428.478
Skew:                          -1.237   Prob(JB):                         0.00
Kurtosis:                       5.043   Cond. No.                     6.25e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is co

In [26]:
#### OLS WITH RF SELECTED FEATS

# LA

# Subset the data to include only the selected features
X_selected = X_lcap[selected_feat_lcap_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_lcap, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     LA   R-squared:                       0.504
Model:                            OLS   Adj. R-squared:                  0.503
Method:                 Least Squares   F-statistic:                     338.0
Date:                Fri, 04 Apr 2025   Prob (F-statistic):               0.00
Time:                        17:53:50   Log-Likelihood:                 11405.
No. Observations:               10327   AIC:                        -2.275e+04
Df Residuals:                   10295   BIC:                        -2.251e+04
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.1985      0.216     -5.553      0.000      -1.622      -0.775
HRk           -0.0025      0.000     -7.045      0.000      -0.003      -0.002
HAvAge         0.0041      0.001      4.907      0.000       0.002       0.006
HW            -0.0006      0.000     -1.611      0.107      -0.001       0.000
HPTS%         -0.2968      0.048     -6.235      0.000      -0.390      -0.203
HSRS           0.1478      0.025      6.009      0.000       0.100       0.196
HSOS          -0.1955      0.033     -5.912      0.000      -0.260      -0.131
HGF/G         -0.1038      0.024     -4.373      0.000      -0.150      -0.057
HGA/G          0.0655      0.023      2.800      0.005       0.020       0.111
HPP            0.0001      0.000      0.951      0.342      -0.000       0.000
HPPA           0.0009      0.000      5.820      0.000       0.001       0.001
HPPOA         -0.0002   6.08e-05     -3.008      0.003      -0.000   -6.37e-05
HSH           -0.0005      0.000     -1.502      0.133      -0.001       0.000
HPIM/G         0.0046      0.001      3.354      0.001       0.002       0.007
HoPIM/G       -0.0017      0.001     -1.298      0.194      -0.004       0.001
HS%            0.0089      0.002      5.766      0.000       0.006       0.012
VAvAge     -8.602e-05      0.001     -0.105      0.916      -0.002       0.002
VSOS           0.0139      0.025      0.564      0.573      -0.035       0.062
VGF/G         -0.0016      0.005     -0.317      0.751      -0.011       0.008
VGA/G         -0.0059      0.005     -1.204      0.228      -0.016       0.004
VPP         -5.03e-05      0.000     -0.355      0.722      -0.000       0.000
VPPO           0.0002   7.98e-05      2.715      0.007    6.02e-05       0.000
VPPA           0.0006      0.001      1.049      0.294      -0.000       0.002
VPPOA      -4.826e-07      0.000     -0.004      0.997      -0.000       0.000
VPK%           0.0009      0.001      0.752      0.452      -0.002       0.003
VSH            0.0005      0.000      1.700      0.089   -7.77e-05       0.001
VPIM/G        -0.0003      0.002     -0.190      0.849      -0.004       0.003
VoPIM/G       -0.0003      0.002     -0.156      0.876      -0.004       0.003
VS%            0.0011      0.002      0.708      0.479      -0.002       0.004
VSV%           0.2232      0.159      1.408      0.159      -0.088       0.534
SDAY           0.0196      0.002     10.609      0.000       0.016       0.023
LCAP           1.0950      0.012     91.026      0.000       1.071       1.119
==============================================================================
Omnibus:                     2747.471   Durbin-Watson:                   1.984
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             6830.031
Skew:                          -1.464   Prob(JB):                         0.00
Ku